# ECMM422J Coursework 1 — ECG Heartbeat Classification with a 1D CNN

**Author:**  Arush Kumar Vishwakarma
**Student ID:** 

## Overview
This notebook builds a 1D Convolutional Neural Network to classify ECG heartbeats from the MIT-BIH Arrhythmia dataset into five categories: Normal, Supraventricular, Ventricular, Fusion, and Unknown/Paced.

The pipeline runs in 9 stages:
1. Setup & reproducibility
2. Load data
3. Preprocessing
4. Visualisation
5. Model architecture
6. Class weight calculation
7. K-fold cross-validation + hyperparameter search
8. Final training
9. Evaluation

---
## Stage 1 — Setup and Reproducibility

Import libraries and lock random seeds so the experiment produces identical results every time.

In [1]:
# Stage 1: Setup and Reproducibility

# Standard library
import os
import random
import json
import time

# Numerical and data libraries
import numpy as np
import pandas as pd

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning library
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import to_categorical, plot_model

# Classical ML utilities from scikit-learn
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

# ---- Reproducibility: lock random seeds ----
# Without this, every training run gives slightly different numbers
# because weight initialisation, data shuffling, and dropout are random.
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Plot styling ----
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

# ---- Confirm environment ----
print('TensorFlow version:', tf.__version__)
print('Devices available:', tf.config.list_physical_devices())
print('Seed locked to:', SEED)

TensorFlow version: 2.21.0
Devices available: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Seed locked to: 42


---
## Stage 2 — Load the Data

The training and test sets are provided as separate CSV files. Each row is one heartbeat:
the first 187 columns are time-domain ECG samples (already normalised to ~[0,1] by the
dataset authors), and the last column is the integer class label in {0, 1, 2, 3, 4}.

No header row, so we pass `header=None` to pandas.

In [2]:
# Stage 2: Load the training and test data

# File paths - data/ folder sits next to the notebook
TRAIN_PATH = 'data/mitbih_train.csv'
TEST_PATH  = 'data/mitbih_test.csv'

# Read both CSVs. header=None because the files have no column names.
t0 = time.time()
train_df = pd.read_csv(TRAIN_PATH, header=None)
test_df  = pd.read_csv(TEST_PATH,  header=None)
print(f'Loaded both CSVs in {time.time() - t0:.1f}s')

# Split each DataFrame into features (X) and label (y).
# .iloc[:, :-1] = "all rows, all columns except the last" = the 187 features
# .iloc[:, -1]  = "all rows, the last column only"        = the class label
# .values gives us numpy arrays (faster than DataFrame for ML).
# astype enforces dtypes: float32 for features (TF prefers this), int64 for labels.
X_train = train_df.iloc[:, :-1].values.astype(np.float32)
y_train = train_df.iloc[:, -1].values.astype(np.int64)

X_test = test_df.iloc[:, :-1].values.astype(np.float32)
y_test = test_df.iloc[:, -1].values.astype(np.int64)

# Sanity prints - shapes, label range, value range
print(f'\nShapes:')
print(f'  X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'  X_test : {X_test.shape}   y_test : {y_test.shape}')

print(f'\nLabel values present (train): {np.unique(y_train)}')
print(f'Label values present (test) : {np.unique(y_test)}')

print(f'\nFeature value range:')
print(f'  Train: [{X_train.min():.3f}, {X_train.max():.3f}]')
print(f'  Test : [{X_test.min():.3f}, {X_test.max():.3f}]')

Loaded both CSVs in 2.6s

Shapes:
  X_train: (87554, 187)   y_train: (87554,)
  X_test : (21892, 187)   y_test : (21892,)

Label values present (train): [0 1 2 3 4]
Label values present (test) : [0 1 2 3 4]

Feature value range:
  Train: [0.000, 1.000]
  Test : [0.000, 1.000]


---
## Stage 3 — Preprocessing

### 3a. Missing values (per class)

The brief requires per-class handling of missing values: if any cell is missing, fill it
with the mean of that column **computed only within the same class**, so we don't blur
class-specific waveform morphology with a global average.

In [4]:
# Stage 3a: Check for and (if needed) impute missing values per class

# Count NaNs in each set
n_missing_train = int(np.isnan(X_train).sum())
n_missing_test  = int(np.isnan(X_test).sum())
print(f'Missing values in X_train: {n_missing_train}')
print(f'Missing values in X_test : {n_missing_test}')

# Per-class mean imputation. Runs only if there are any NaNs to fix.
def impute_per_class(X, y):
    """
    For each class c:
      compute the column-wise mean across that class (ignoring NaN),
      then fill in any NaN values in that class's rows with those means.
    Returns a new array (does not modify the input).
    """
    X = X.copy()
    for c in np.unique(y):
        mask = (y == c)                                 # rows belonging to class c
        col_means = np.nanmean(X[mask], axis=0)         # per-column mean for class c
        # find positions where there's a NaN AND the row is in class c
        rows, cols = np.where(np.isnan(X) & mask[:, None])
        X[rows, cols] = col_means[cols]
    return X

if n_missing_train > 0:
    X_train = impute_per_class(X_train, y_train)
    print(f'After imputation, train NaNs: {int(np.isnan(X_train).sum())}')
else:
    print('No imputation needed for train.')

if n_missing_test > 0:
    X_test = impute_per_class(X_test, y_test)
    print(f'After imputation, test NaNs: {int(np.isnan(X_test).sum())}')
else:
    print('No imputation needed for test.')

Missing values in X_train: 0
Missing values in X_test : 0
No imputation needed for train.
No imputation needed for test.


### 3b. Normalisation check

The dataset authors (Kachuee et al., 2018) pre-normalised each beat by dividing by its
peak amplitude, so values are already in [0, 1]. We confirm this rather than re-scale
(re-scaling already-scaled data introduces unnecessary distortion).

In [5]:
# Stage 3b: Confirm features are in [0, 1] (already normalised by dataset authors)

print('Training set:')
print(f'  min  = {X_train.min():.4f}')
print(f'  max  = {X_train.max():.4f}')
print(f'  mean = {X_train.mean():.4f}')
print(f'  std  = {X_train.std():.4f}')

print('\nTest set:')
print(f'  min  = {X_test.min():.4f}')
print(f'  max  = {X_test.max():.4f}')
print(f'  mean = {X_test.mean():.4f}')
print(f'  std  = {X_test.std():.4f}')

print('\nNo further scaling applied.')

Training set:
  min  = 0.0000
  max  = 1.0000
  mean = 0.1743
  std  = 0.2263

Test set:
  min  = 0.0000
  max  = 1.0000
  mean = 0.1735
  std  = 0.2256

No further scaling applied.


### 3c. Reshape for Conv1D and one-hot encode labels

Conv1D expects input shape `(batch, timesteps, channels)`, so we reshape from `(N, 187)`
to `(N, 187, 1)`. The trailing `1` is the channel dimension (ECG has a single channel).

For labels, we one-hot encode: class 2 → `[0, 0, 1, 0, 0]`. This pairs with the softmax
output and categorical cross-entropy loss, and tells the model the classes are categorical
rather than ordered.

In [6]:
# Stage 3c: Reshape inputs for Conv1D and one-hot encode labels

NUM_CLASSES = 5
TIMESTEPS = X_train.shape[1]   # 187 — read from data rather than hard-code

# Reshape (N, 187) -> (N, 187, 1) so Conv1D treats each beat as a single-channel signal
X_train = X_train.reshape(-1, TIMESTEPS, 1)
X_test  = X_test.reshape(-1, TIMESTEPS, 1)

# One-hot encode labels: integer class -> 5-element binary vector
y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_test_oh  = to_categorical(y_test,  NUM_CLASSES)

# Sanity prints
print(f'TIMESTEPS = {TIMESTEPS}')
print(f'NUM_CLASSES = {NUM_CLASSES}')
print()
print(f'X_train reshaped to: {X_train.shape}')
print(f'X_test  reshaped to: {X_test.shape}')
print()
print(f'y_train one-hot: {y_train_oh.shape}')
print(f'y_test  one-hot: {y_test_oh.shape}')
print()
print(f'Example label conversion:')
print(f'  Integer label: {y_train[0]}')
print(f'  One-hot label: {y_train_oh[0]}')

TIMESTEPS = 187
NUM_CLASSES = 5

X_train reshaped to: (87554, 187, 1)
X_test  reshaped to: (21892, 187, 1)

y_train one-hot: (87554, 5)
y_test  one-hot: (21892, 5)

Example label conversion:
  Integer label: 0
  One-hot label: [1. 0. 0. 0. 0.]
